# FingerPrint-Based Multi-Layer Perceptron

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import pandas as pd
import numpy as np
from pyprojroot import here
import joblib
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge as KR
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from dl_hplc_smrt.data_transformers import SmilesToMolTransformer as SMT
from dl_hplc_smrt.data_transformers import MolToFingerPrintTransformer as MFPT
from dl_hplc_smrt.models import ComplexMultilayerPerceptron as CMLP
from dl_hplc_smrt.utils import OptimizeParametersValidation as OPV
from dl_hplc_smrt.utils import evaluate_sklearn_model, plot_results

In [24]:
sns.set_style("whitegrid")

In [25]:
RAW_DATA_PATH = here() / "data" / "raw"
PROCESSED_DATA_PATH = here() / "data" / "processed"

### Read the SMILES Sets (Train, Validation, Test)

In [26]:
x_train_ret = np.load(PROCESSED_DATA_PATH / "X_train_set_retained.npy", allow_pickle=True).reshape(-1,1)
x_val_ret = np.load(PROCESSED_DATA_PATH / "X_val_set_retained.npy", allow_pickle=True).reshape(-1,1)
x_test_ret = np.load(PROCESSED_DATA_PATH / "X_test_set_retained.npy", allow_pickle=True).reshape(-1,1)
print(x_train_ret.shape, x_val_ret.shape, x_test_ret.shape)
x_train_ret[0:5,:]


(59593, 1) (6623, 1) (10710, 1)


array([['COc1ccc(CC(O)=NC2CCN(c3nnc4ccccn34)CC2)cc1'],
       ['CCc1noc(N=C(C)O)c1-c1ccc(OC)c(S(=O)(=O)N2CC=C(c3ccccc3)CC2)c1'],
       ['CN(C)CCCN=C(O)C1CCN(c2nc3ccc(-c4cccnc4)nc3s2)CC1'],
       ['CCOC(=O)c1c(S(=O)(=O)N2CCc3ccccc3C2)c(C)n(C)c1C'],
       ['Cc1cc(C)c(N=C(O)Cn2cnc3nc(N4CCC(C(O)=NC5CC5)CC4)sc3c2=O)c(C)c1']],
      dtype=object)

In [27]:
y_train_ret = np.load(PROCESSED_DATA_PATH / "y_train_set_retained.npy", allow_pickle=True)
y_val_ret = np.load(PROCESSED_DATA_PATH / "y_val_set_retained.npy", allow_pickle=True)
y_test_ret = np.load(PROCESSED_DATA_PATH / "y_test_set_retained.npy", allow_pickle=True)
print(y_train_ret.shape, y_val_ret.shape, y_test_ret.shape)
y_train_ret[0:5]

(59593,) (6623,) (10710,)


array([626.4, 917.6, 543.4, 989.2, 754.4])

#### Convert SMILES to mol

In [28]:
print(x_train_ret.shape)
fp_transformer = SMT().fit(x_train_ret)
x_train_ret_processed = fp_transformer.transform(x_train_ret)
print(x_train_ret_processed.shape)

(59593, 1)
(59593, 1)


#### Convert to fingerprints (Morgan, for example)

In [29]:
print(x_train_ret_processed.shape)
fp_transformer = MFPT(fp_type="morgan", radius=5, fp_size=1024, dense=True, counts=True).fit(x_train_ret_processed)
x_train_ret_processed = fp_transformer.transform(x_train_ret_processed)
print(x_train_ret_processed.shape)

(59593, 1)
(59593, 1024)


#### Normalize the variance

In [30]:
print(x_train_ret_processed.shape)
standard_scaler = StandardScaler(with_mean=True).fit(x_train_ret_processed)
x_train_ret_processed = standard_scaler.transform(x_train_ret_processed)
print(x_train_ret_processed.shape)

(59593, 1024)
(59593, 1024)


#### Filter by variance (let's remove those that are always zero or very unusual)

In [31]:
print(x_train_ret_processed.shape)
variance_thres = VarianceThreshold(threshold=0.0).fit(x_train_ret_processed)
x_train_ret_processed = variance_thres.transform(x_train_ret_processed)
print(x_train_ret_processed.shape)

(59593, 1024)
(59593, 1024)


In [32]:
x_train_ret_processed[:5,:5]

array([[-0.2147815 , -0.45340656, -0.3312962 ,  3.09786684, -0.54274712],
       [-0.2147815 , -0.45340656, -0.3312962 , -0.28087689, -0.54274712],
       [-0.2147815 ,  1.75500025, -0.3312962 , -0.28087689, -0.54274712],
       [-0.2147815 , -0.45340656, -0.3312962 , -0.28087689, -0.54274712],
       [-0.2147815 , -0.45340656, -0.3312962 , -0.28087689, -0.54274712]])

### First Tests: Morgan Fingerprints Based on Bits 

Let's Write a Data Processing Pipleline

In [33]:
fp_based_preparation = Pipeline([
    ("mol_converter", SMT()),
    ("fp_transformer", MFPT(fp_type="morgan", radius=2, fp_size=1024, dense=True, counts=False)),
    ("standard_scaler", StandardScaler(with_mean=False)),
    ("variance_thres", VarianceThreshold(threshold=0.0)),# Remove constant all-zero features    
    ])

In [34]:
x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

In [35]:
print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

(59593, 1024)
(6623, 1024)
(10710, 1024)


# Comparing Different Fingerprints and Models

We will use these scores

In [36]:
scores={
    "r**2": r2_score,
    "mae": mean_absolute_error,
    "mse": mean_squared_error,
    "rmse": root_mean_squared_error
}

### MACCS Bits

Change the parameters of the data transformation pipeline

In [37]:
new_params = {
    "fp_transformer__fp_type": "maccs",
    "fp_transformer__counts": False
}
fp_based_preparation.set_params(**new_params)

x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

(59593, 152)
(6623, 152)
(10710, 152)


In [38]:
datasets={
    "train": [x_train_ret_processed, y_train_ret],
    "validation": [x_val_ret_processed, y_val_ret]
}

Ridge regression model

In [39]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=ridge_model, datasets=datasets, scores=scores)

,dataset,r**2,mae,mse,rmse
0,train,0.405104,104.938432,18203.918873,134.921899
1,validation,0.396676,105.136378,18452.572310,135.840246


Random Forest model

In [40]:
rfr_model = RFR(n_estimators=300, max_depth=25, max_features=0.7, max_samples=0.7, n_jobs=6)
rfr_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=rfr_model, datasets=datasets, scores=scores)

,dataset,r**2,mae,mse,rmse
0,train,0.905424,38.231967,2894.051452,53.796389
1,validation,0.680830,69.593392,9761.766415,98.801652


### Morgan Bits

In [41]:
new_params = {
    "fp_transformer__fp_type": "morgan",
    "fp_transformer__counts": False,
    "fp_transformer__fp_size": 1024,
    "fp_transformer__radius": 2

}
fp_based_preparation.set_params(**new_params)

x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

(59593, 1024)
(6623, 1024)
(10710, 1024)


In [42]:
datasets={
    "train": [x_train_ret_processed, y_train_ret],
    "validation": [x_val_ret_processed, y_val_ret]
}

Ridge regression model

In [43]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=ridge_model, datasets=datasets, scores=scores)

,dataset,r**2,mae,mse,rmse
0,train,0.648713,76.800651,10749.440278,103.679508
1,validation,0.610897,79.695675,11900.646993,109.090087


Random Forest model

In [ ]:
rfr_model = RFR(n_estimators=300, max_depth=25, max_features=0.7, max_samples=0.7, n_jobs=6)
rfr_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=rfr_model, datasets=datasets, scores=scores)

### Morgan Counts

In [ ]:
new_params = {
    "fp_transformer__fp_type": "morgan",
    "fp_transformer__counts": True,
    "fp_transformer__fp_size": 1024,
    "fp_transformer__radius": 2

}
fp_based_preparation.set_params(**new_params)

x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

In [ ]:
datasets={
    "train": [x_train_ret_processed, y_train_ret],
    "validation": [x_val_ret_processed, y_val_ret]
}

Ridge regression model

In [ ]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=ridge_model, datasets=datasets, scores=scores)

Random Forest model

In [ ]:
rfr_model = RFR(n_estimators=300, max_depth=25, max_features=0.7, max_samples=0.7, n_jobs=6)
rfr_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=rfr_model, datasets=datasets, scores=scores)

### RDKit Bits

In [ ]:
new_params = {
    "fp_transformer__fp_type": "rdkit",
    "fp_transformer__counts": False,
    "fp_transformer__fp_size": 1024,
    "fp_transformer__radius": 5

}
fp_based_preparation.set_params(**new_params)

x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

In [ ]:
datasets={
    "train": [x_train_ret_processed, y_train_ret],
    "validation": [x_val_ret_processed, y_val_ret]
}

Ridge regression model

In [ ]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=ridge_model, datasets=datasets, scores=scores)

Random Forest model

In [ ]:
rfr_model = RFR(n_estimators=300, max_depth=25, max_features=0.7, max_samples=0.7, n_jobs=6)
rfr_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=rfr_model, datasets=datasets, scores=scores)

### RDKit Counts

In [ ]:
new_params = {
    "fp_transformer__fp_type": "rdkit",
    "fp_transformer__counts": True,
    "fp_transformer__fp_size": 1024,
    "fp_transformer__radius": 5

}
fp_based_preparation.set_params(**new_params)

x_train_ret_processed = fp_based_preparation.fit_transform(x_train_ret)
x_val_ret_processed = fp_based_preparation.transform(x_val_ret)
x_test_ret_processed = fp_based_preparation.transform(x_test_ret)

print(x_train_ret_processed.shape)
print(x_val_ret_processed.shape)
print(x_test_ret_processed.shape)

In [ ]:
datasets={
    "train": [x_train_ret_processed, y_train_ret],
    "validation": [x_val_ret_processed, y_val_ret]
}

Ridge regression model

In [ ]:
ridge_model = Ridge(alpha=1)
ridge_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=ridge_model, datasets=datasets, scores=scores)

Random Forest model

In [ ]:
rfr_model = RFR(n_estimators=300, max_depth=25, max_features=0.7, max_samples=0.7, n_jobs=6)
rfr_model.fit(X=x_train_ret_processed, y=y_train_ret)
evaluate_sklearn_model(model=rfr_model, datasets=datasets, scores=scores)

### Conclussions

1. For all fingerprints considered, there is model performance to be gain by using Random Forest instead of Ridge regression. Modeling the nonlionear behavior is important.
2. RdKit counts appear to be the best option to build our modeks based on fingerprints.
3. RF models appear to be strongly overfitted, it should be possible to improve them by include regularization, tuning the hyperparameters

## Base Model: Random Forest

So our baseline model will Random Forest based on RDKit counts. Let's try to get our best model by tuning the hyperparameters

Let's prepare a new pipeline including the model

In [ ]:
rfr_rdkit_pipeline = Pipeline([    
    ("mol_converter", SMT()),
    ("fp_transformer", MFPT(fp_type="rdkit", radius=5, fp_size=1024, dense=True, counts=True)),
    ("standard_scaler", StandardScaler(with_mean=False)),
    ("variance_thres", VarianceThreshold(threshold=0.0)),# Remove constant all-zero features    
    ("rfr", RFR(n_jobs=6))
    ])

Let's do a first search with 0.70 of the data

In [ ]:
x_temp_1, x_temp_2, y_temp_1, y_temp_2 = train_test_split(x_train_ret, y_train_ret, train_size=0.70)

First, let's tune the fp radius

In [ ]:
rfr_param_grid = {
    'fp_transformer__radius': [1,3,5,7,9],
}
rfr_hyperparam_tuning = OPV(model=rfr_rdkit_pipeline, param_grid=rfr_param_grid, scoring=mean_absolute_error, keep_highest=False, keep_lowest=False).fit(x_temp_1, y_temp_1, x_val_ret, y_val_ret)
results = rfr_hyperparam_tuning.get_results()
results.sort_values(by = "validation score").head(10)

Three seems to be the sweet spot

First, let's tune max_features and max_samples together

In [ ]:
rfr_param_grid = {
    'rfr__max_features': [0.25, 0.5, 0.7, 0.9],
    'rfr__max_samples': [0.25, 0.5, 0.7, 0.9],
}
rfr_rdkit_pipeline.set_params(**rfr_hyperparam_tuning.lowest_params_)
rfr_hyperparam_tuning = OPV(model=rfr_rdkit_pipeline, param_grid=rfr_param_grid, scoring=mean_absolute_error, keep_highest=False, keep_lowest=False).fit(x_temp_1, y_temp_1, x_val_ret, y_val_ret)
results = rfr_hyperparam_tuning.get_results()
results.sort_values(by = "validation score").head(10)

Clearly, the most important factor is to set rfr__max_samples to 0.9. rfr__max_features will be set to 0.7

Now it is the turn of the max_depth

In [ ]:
rfr_param_grid = {
    'rfr__max_depth': [5, 25, 50, 100, 200, None],
}
rfr_rdkit_pipeline.set_params(**rfr_hyperparam_tuning.lowest_params_)
rfr_hyperparam_tuning = OPV(model=rfr_rdkit_pipeline, param_grid=rfr_param_grid, scoring=mean_absolute_error, keep_highest=False, keep_lowest=False).fit(x_temp_1, y_temp_1, x_val_ret, y_val_ret)
results = rfr_hyperparam_tuning.get_results()
results.sort_values(by = "validation score").head(10)

Very deep trees! 100 is selected as the best option

Let's modify our pipeline to include best parameters so far, work with the whole dataset and evaluate the current best model

In [ ]:
rfr_rdkit_pipeline = Pipeline([    
    ("mol_converter", SMT()),
    ("fp_transformer", MFPT(fp_type="rdkit", radius=3, fp_size=1024, dense=True, counts=True)),
    ("standard_scaler", StandardScaler(with_mean=False)),
    ("variance_thres", VarianceThreshold(threshold=0.0)),# Remove constant all-zero features    
    ("rfr", RFR(n_estimators=300, max_depth=100, max_features=0.7, max_samples=0.9, n_jobs=7, random_state=123456))
    ])

rfr_rdkit_pipeline.fit(x_train_ret, y_train_ret)
datasets={
    "train": [x_train_ret, y_train_ret],
    "validation": [x_val_ret, y_val_ret]
}
evaluate_sklearn_model(model=rfr_rdkit_pipeline, datasets=datasets, scores=scores)

In [ ]:
my_fig = plot_results(y_val_ret, rfr_rdkit_pipeline.predict(x_val_ret))

## Save Baseline Model

In [ ]:
joblib.dump(rfr_rdkit_pipeline, here() / "models" / "random_forest_rdkit_pipeline_2026_06_29_v0.pkl", compress=3)